# Tutorial 7 · Transformer applications: language models and diffusion
**CSCI3230 / ESTR3108 · 45-minute guided tutorial**

The lecture covers Transformer operations. Here we briefly revisit attention, then explore
**two separate applications**: an autoregressive language model and an image diffusion model
whose denoising predictor is a Transformer.

By the end, you should be able to:
1. Read an attention row and explain what a causal mask prevents.
2. Connect next-token training targets to the text-generation loop.
3. Change a decoding setting and explain what changed—and what did not.
4. Explain the job of a Transformer inside a diffusion model.
5. Distinguish the predictions and inference loops of these two applications.

| Time | Topic | Activity |
|---|---|---|
| 0–4 min | Quick Transformer recap | Follow how attention combines information |
| 4–25 min | Application 1: language modeling | Learn causal masking; generate text; change temperature |
| 25–42 min | Application 2: image diffusion | Follow noise levels, patch tokens and real sampling trajectories |
| 42–45 min | Check understanding | Compare the two applications; answer exit questions |

**Predict → run → explain.** Student controls are marked **TRY**. Model and visualization
helpers can stay collapsed; you do not need to read every implementation during class.
The optional extensions and training appendix are **outside the 45 minutes**.

## Setup · Run once before class starts

Use local Jupyter or Google Colab with **PyTorch 2.0+**, NumPy and Matplotlib.
For local setup: `pip install torch numpy matplotlib jupyter`.
`torchvision` is needed only for the optional MNIST training appendix.

This notebook is self-contained: no repository clone, external Python modules, API key,
or dataset download is needed for the core calculations. Ten MNIST test images are embedded.
The website activity needs Internet access; the local demonstrations also work offline.

Put the TA's files in the notebook's **working directory**:

| File | Purpose |
|---|---|
| `tutorial7_lm.pt` | Pretrained TinyLM for text-generation experiments |
| `tutorial7_dit_demo.npz` | Saved **actual model** sampling trajectories and comparison images |
| `tutorial7_dit.pt` (optional for students) | Pretrained TinyDiT for new live image samples |

Without weights, shape and attention examples still run. TinyLM is explicitly marked
**UNTRAINED**; its random text is not evidence of language learning. Without a diffusion
demo file, the trained sampling section prints preparation instructions instead of inventing results.
**Run all never starts training or a full diffusion sampling run by default.**

In [ ]:
from pathlib import Path
import base64
import copy
import hashlib
import math
import random
import textwrap
import time
import zlib

import numpy as np
import matplotlib.pyplot as plt
from matplotlib.animation import FuncAnimation
from matplotlib.patches import Rectangle
from IPython.display import HTML, display
import torch
from torch import nn
import torch.nn.functional as F

torch.manual_seed(42)
torch.set_num_threads(2)
plt.rcParams.update({
    'figure.dpi': 100, 'font.size': 11,
    'axes.spines.top': False, 'axes.spines.right': False,
    'animation.embed_limit': 40,
})
PURPLE, BLUE, GREEN, ORANGE = '#751475', '#2878a5', '#20846d', '#bf7b19'
LM_PATH = Path('tutorial7_lm.pt')
DIT_PATH = Path('tutorial7_dit.pt')
DEMO_PATH = Path('tutorial7_dit_demo.npz')
SCHEMA = 'csci3230_tutorial7_v1'
print('PyTorch:', torch.__version__, '| Classroom inference: CPU')
print('Working directory:', Path.cwd())
for path in (LM_PATH, DEMO_PATH, DIT_PATH):
    print(('FOUND   ' if path.exists() else 'MISSING '), path)

**Colab only:** upload the provided files using the left-hand Files panel, or enable the cell
below once. Local Jupyter users can copy the files next to the notebook.

In [ ]:
UPLOAD_FILES = False
if UPLOAD_FILES:
    from google.colab import files
    files.upload()
else:
    print('Upload skipped.')

In [ ]:
# Embedded real MNIST test images: one example of each class, ordered 0–9.
# Test indices: [3, 2, 1, 18, 4, 8, 11, 0, 61, 7]. These are not training data.
_pixels = zlib.decompress(base64.b64decode('eJztmXtUTVkcx8+gKKFUyFAt0aAhzzF5DmHyiBkKMxjWPMwyBoswEsqMskaPQdw8ZmkMk8ojMwyT51hIXiUt41l5lJK0inSrs3/7zj733nPds8/5rfnHWrM8vmvdfX7dz/3t995n/3aC8P+psQ7OeyCsgyjCLG3kegZlc46LjKXMHqjBQJSYKN7uqUIHDcBUms8SHg3Kk/ziAwdGiOJMJfIsZtnd/tFeEDweiE/n2fDVP+piNGezX3lx7Jy7yfTI5NnzGnieN8AOaxYjirKpyvOGzFwHsVo9cNdka1hL8gYIWuxgHmMHlO27CTBiRCGAsXOUSJhn7EpTEs8x1hsm9uCYlz3HhIFxJqY9fgF7xT0fBrhrstdD3nQ2yiaScShbVYmiLlU6lAXRQSg7X9AYQ570Ouo2jZ5CWQwNxJDf40uNMLbU8Bua5S76MYZalVxD3RbTRJQl0DiUFdIhGBog4iyWXqqPIPtrNBRzs8nYp1pBr5IWLNtlMOimaqEUMOqmxmYgoatx+wCWqFCvOrji6SDYZkGMigWKV9zYY0kNDFbn6dFcSnNAixm1UA8ZyFiM1kMxtowiANYgaF81JDpoI7dSeOiljYQMgFgEjamBY0iOzpm4WxTAHsRNqAFwQ5DEfFyYbAQbF5cOOp0u3t6aGZX8U5LZCrOwvfBctXp9ysKFfs8zXRQWFia5bAkL64QV/NrJodO6dd0QtIJ1Vt325los0tTHRcM12FdA1s6Y9Rj0EepZ/SfsZOmARwDbbXhmAB/p0fckwI4GHDsCnsZnnzKACRxbx9iXxydOnBhvPRVM+gzGdJQnTEFQMwVrOsXBq8IyXao+4Ws7att+C83x4Wtb39HxHW9Hx7VPAdKVHejSV7b6slV/yBoF5unHynaTW1ARYMU+1QOxzNbeIpywdlwA0E+2vyCwVtGOo5C31Vuy5uZWg5IJDnnVINYySa3IdOYaMe1Qoal9p5a25BvI3nIdVo9cPqlDQzV5fdXjjtkY3lbFQovNxvpkHjU4JbPpV/izzzASZbbmE1cl6lJ2Q94+/+ZZsr632WpuACULepIrm7FwTLnQUsg3ZsuzpE55qmh2l8hmFMlVIKEFsUQMKWSnktldzDFvEi1AdfxOIWeDmVbsOEUsJcvqlPqMMJUUi4TYCby6BzEJwjZCVEhWOCFdMBahDrms/VC2ilShrKRsLsr2o0e0V0ROvr6+jiuCfdVk1KYbbA+5Vq3qca+4KmLZQzk2yPz11VQmq+9dVgYIfuX3y5OXDnPhlkrjLBjD5q3gXk9VBdvfYSVyiHKIhIfNtJEwBQraIEjQwV4MCaVQHd4dYVJUK8ZPDg3u3DmYzzza6oBSwu0S9XvfzBdlSJaq8vUPyDTDNI1SF0Ptxp47tFkP5nOUDcR6DWa3U8qxLk0zImt5oBjyIrSIpKkbWmDopVfDi7APQzpCliFsITn8PpZlLFmEFrdZ3wNDreE06qazML/gYG8luwvTjc+Ee08JKVdU2L7oHksbvFcEhtK0u4ZC66uyOYTF9K0jCLkfxSbvH8S6OYlknlQmOexjao81S2fM+zEk2Br/ij1va8UyYL4QD9vltp6xrssZ5rebHJXbuotnzsX6JcZjS2aVdXGt86W6uF0gJ5sIozNrleORDvulvWC0V9vNUMUNVZscOD3Oz++HMwT+UcX3btfZ6wjY52f+pMTk+HV0RUx0dEc1eaMXqqD0xIT+7bVZPmWqzDArtZc1858fMH/7fXqX/aKuiFJ1nO40pKm/v38/1zLKn0QsGg85mgEPU4uHhvGY2wpa/i6C+tVSrUtjoyLpEVUYZJbdpZq+CBKW04MYGiVW+CHIOY8mIaj+BXoLuxXwpujVmscdGvIWwiIp7YWgAU9wFkrpLWyFhdJsbGxeJvn4ahyhjLL7loVpl7OzszcMbsKjvyh7TZs+V5X3Jn0uUHo2PLz/1KChCfl0m4KtBjgrb2VbIJdnTmZzZCXHZgLMMFntigB+UdZlAy2Shsh7E1toB7g7HOcMmm3XLoAF+Hei+TYITrlwrYJCaYQTTwRhBisHCgNVPsLwVGrUdzxo/f0D1h/pIR/kQg0XGyWVAOh/7dqk4eRnQLkXMosL4MrU8MSLUnn8uTCqkhpMxRW2U9Xk7ZCYrKys7Kww7DT5Ri9WTTecxLbHyXcANN7Rkto8YtMiSXunWGNcQuUhtmrkUQmX0xksbqVmY+lJodHnt6jhnDrbCfARSw8CPaG+6UyDLYIULNBV6jwnwOWOwUniY1rWWcWal0v1TG9/HTaqHYdWGOjaRkIULdDYsodujWPVsEuDbWpm1iS4h26j9ZJgOerY7Rl4ozCE7laH8Ga53qRdUUd3iv8LRjhcpe4cWU0LxuCO/61/AQAq+aU='))
digit_images = torch.from_numpy(np.frombuffer(_pixels, np.uint8).copy()).reshape(10, 1, 28, 28).float() / 255
clean_images = digit_images * 2 - 1  # Diffusion convention: clean pixels in [-1, 1].

In [ ]:
# Plot helpers only. Closing each figure prevents a duplicate static plot below an animation.
def animation_html(fig, draw, frames, interval=650):
    animation = FuncAnimation(fig, draw, frames=frames, interval=interval, repeat=False)
    try:
        html = animation.to_jshtml(default_mode='once')
    finally:
        plt.close(fig)
    return HTML(html)


def image_grid(images, titles, columns=4, *, heading=None):
    images = torch.as_tensor(images).detach().cpu()
    rows = math.ceil(len(images) / columns)
    fig, axes = plt.subplots(rows, columns, figsize=(2.1 * columns, 2.2 * rows), squeeze=False)
    for ax in axes.flat:
        ax.axis('off')
    for ax, image, title in zip(axes.flat, images, titles):
        ax.imshow(image.squeeze(), cmap='gray', vmin=-1, vmax=1)
        ax.set_title(title, fontsize=10)
    if heading:
        fig.suptitle(heading)
    fig.tight_layout()
    plt.show()


def visible_character(char):
    return {' ': 'space', '\n': r'\n'}.get(char, char)


def count_parameters(model):
    return sum(parameter.numel() for parameter in model.parameters())


def file_sha256(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for chunk in iter(lambda: stream.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

## 1 · Quick recap: how does attention combine information? (0–4 min)

Each **query position** receives a weighted mixture of **value vectors**. One row of the
attention matrix belongs to one query; one column belongs to one key/value position.
Each row sums to 1.

$$A=\operatorname{softmax}(QK^T/\sqrt{d_k}),\qquad O=AV.$$

Here every position may read every position. We will introduce a task-specific restriction
later, when we build a language model; no knowledge of that restriction is assumed here.

The animation uses **hand-chosen numbers**, not learned linguistic relationships.
Scalar values make the weighted sum visible; real heads use value vectors.

**Predict first:** is the output simply the value with the largest attention weight,
or a weighted combination of all the values?

In [ ]:
def make_attention_demo():
    names = ['A', 'B', 'C', 'D']
    q = torch.tensor([[1., 0.], [.5, 1.], [-1., .5], [1., 1.]])
    k = torch.tensor([[1., 0.], [0., 1.], [-1., 0.], [1., 1.]])
    values = torch.tensor([2., 4., 8., 6.])
    weights = (q @ k.T / math.sqrt(q.shape[-1])).softmax(-1)
    assert torch.allclose(weights.sum(-1), torch.ones(4))
    fig, axes = plt.subplots(1, 3, figsize=(12, 4.2), gridspec_kw={'width_ratios':[1.1,1,1.5]})
    fig.subplots_adjust(top=.78,bottom=.22,wspace=.45)
    fig.text(.5,.045,'Each query uses its own attention row to form a weighted mixture.',ha='center')
    def draw(query):
        for ax in axes:ax.clear()
        ax,bars,summary=axes
        fig.suptitle(f'Query {names[query]}: read one attention row, then mix values',fontsize=15)
        ax.imshow(weights,cmap='Purples',vmin=0,vmax=1)
        ax.add_patch(Rectangle((-.5,query-.5),4,1,fill=False,ec=ORANGE,lw=3))
        for i in range(4):
            for j in range(4):
                ax.text(j,i,f'{weights[i,j]:.2f}',ha='center',va='center',fontsize=10)
        ax.set(xticks=range(4),xticklabels=names,yticks=range(4),yticklabels=names,
               xlabel='Key / value position',ylabel='Query position',title='Attention weights')
        w=weights[query]
        bars.bar(names,w.numpy(),color=GREEN)
        bars.set(ylim=(0,1.12),ylabel='Weight',title=f'Weights used by {names[query]}')
        for j in range(4):bars.text(j,float(w[j])+.04,f'{w[j]:.2f}',ha='center',fontsize=10)
        summary.axis('off')
        contributions=w*values
        summary.text(0,.96,'\n'.join(f'{names[j]}: {w[j]:.2f} × {values[j]:.0f} = {contributions[j]:.2f}'
                                     for j in range(4)),va='top',family='monospace',fontsize=12)
        summary.text(0,.10,f'Output = {contributions.sum():.2f}',color=PURPLE,fontsize=15)
    return animation_html(fig,draw,frames=4,interval=1500)

display(make_attention_demo())

**Explain:** attention combines values using the full weight vector; it does not simply copy
the value with the largest weight.

A Transformer block also contains a per-token MLP, residual connections and normalization.
We use the implementation below without repeating the lecture derivation.
Our blocks preserve `[B, N, D]`: batch size, number of tokens, and embedding dimension.

In [ ]:
from __future__ import annotations
# Reusable implementation adapted from the course repository; expand only if needed.
"""Small, readable Transformer building blocks for Day 7."""

import math

import torch
from torch import nn
from torch.nn import functional as F


def patchify(images: torch.Tensor, patch_size: int) -> torch.Tensor:
    """Convert `[B, C, H, W]` images to `[B, N, C * P * P]` patches."""

    if images.ndim != 4:
        raise ValueError("images must have shape [B, C, H, W].")
    if patch_size <= 0:
        raise ValueError("patch_size must be positive.")

    batch_size, channels, height, width = images.shape
    if height % patch_size or width % patch_size:
        raise ValueError("image height and width must be divisible by patch_size.")

    grid_height = height // patch_size
    grid_width = width // patch_size
    patches = images.reshape(
        batch_size,
        channels,
        grid_height,
        patch_size,
        grid_width,
        patch_size,
    )
    patches = patches.permute(0, 2, 4, 1, 3, 5)
    return patches.reshape(
        batch_size,
        grid_height * grid_width,
        channels * patch_size * patch_size,
    )


def unpatchify(
    patches: torch.Tensor,
    *,
    image_size: int,
    patch_size: int,
    channels: int = 1,
) -> torch.Tensor:
    """Reconstruct square `[B, C, H, W]` images from flattened patches."""

    if patches.ndim != 3:
        raise ValueError("patches must have shape [B, N, patch_values].")
    if image_size <= 0 or patch_size <= 0:
        raise ValueError("image_size and patch_size must be positive.")
    if image_size % patch_size:
        raise ValueError("image_size must be divisible by patch_size.")

    batch_size, patch_count, patch_values = patches.shape
    grid_size = image_size // patch_size
    expected_patch_count = grid_size * grid_size
    expected_patch_values = channels * patch_size * patch_size
    if patch_count != expected_patch_count:
        raise ValueError(
            f"expected {expected_patch_count} patches, received {patch_count}."
        )
    if patch_values != expected_patch_values:
        raise ValueError(
            f"expected {expected_patch_values} values per patch, "
            f"received {patch_values}."
        )

    images = patches.reshape(
        batch_size,
        grid_size,
        grid_size,
        channels,
        patch_size,
        patch_size,
    )
    images = images.permute(0, 3, 1, 4, 2, 5)
    return images.reshape(batch_size, channels, image_size, image_size)


class MultiHeadSelfAttention(nn.Module):
    """Scaled dot-product self-attention with an optional causal mask."""

    def __init__(self, hidden_dim: int, num_heads: int) -> None:
        super().__init__()
        if hidden_dim <= 0 or num_heads <= 0:
            raise ValueError("hidden_dim and num_heads must be positive.")
        if hidden_dim % num_heads:
            raise ValueError("hidden_dim must be divisible by num_heads.")

        self.hidden_dim = hidden_dim
        self.num_heads = num_heads
        self.head_dim = hidden_dim // num_heads
        self.qkv_projection = nn.Linear(hidden_dim, 3 * hidden_dim)
        self.output_projection = nn.Linear(hidden_dim, hidden_dim)

    def forward(
        self,
        tokens: torch.Tensor,
        *,
        causal: bool = False,
        return_attention: bool = False,
    ) -> torch.Tensor | tuple[torch.Tensor, torch.Tensor]:
        if tokens.ndim != 3 or tokens.shape[-1] != self.hidden_dim:
            raise ValueError(
                f"tokens must have shape [B, N, {self.hidden_dim}]."
            )

        batch_size, token_count, _ = tokens.shape
        qkv = self.qkv_projection(tokens)
        qkv = qkv.reshape(
            batch_size,
            token_count,
            3,
            self.num_heads,
            self.head_dim,
        )
        queries, keys, values = qkv.permute(2, 0, 3, 1, 4).unbind(0)

        if return_attention:
            scores = queries @ keys.transpose(-2, -1)
            scores = scores / math.sqrt(self.head_dim)
            if causal:
                future_positions = torch.ones(
                    token_count,
                    token_count,
                    dtype=torch.bool,
                    device=tokens.device,
                ).triu(diagonal=1)
                scores = scores.masked_fill(future_positions, float("-inf"))
            attention = torch.softmax(scores, dim=-1)
            mixed_values = attention @ values
        else:
            mixed_values = F.scaled_dot_product_attention(
                queries,
                keys,
                values,
                is_causal=causal,
            )
        mixed_values = mixed_values.transpose(1, 2).reshape(
            batch_size,
            token_count,
            self.hidden_dim,
        )
        output = self.output_projection(mixed_values)

        if return_attention:
            return output, attention
        return output


class TransformerBlock(nn.Module):
    """A pre-normalization Transformer block with a per-token MLP."""

    def __init__(
        self,
        hidden_dim: int,
        num_heads: int,
        *,
        mlp_ratio: int = 4,
    ) -> None:
        super().__init__()
        if mlp_ratio <= 0:
            raise ValueError("mlp_ratio must be positive.")

        self.attention_norm = nn.LayerNorm(hidden_dim)
        self.attention = MultiHeadSelfAttention(hidden_dim, num_heads)
        self.mlp_norm = nn.LayerNorm(hidden_dim)
        self.mlp = nn.Sequential(
            nn.Linear(hidden_dim, mlp_ratio * hidden_dim),
            nn.GELU(),
            nn.Linear(mlp_ratio * hidden_dim, hidden_dim),
        )

    def forward(
        self,
        tokens: torch.Tensor,
        *,
        causal: bool = False,
        return_attention: bool = False,
    ) -> torch.Tensor | tuple[torch.Tensor, torch.Tensor]:
        normalized_tokens = self.attention_norm(tokens)
        attention_result = self.attention(
            normalized_tokens,
            causal=causal,
            return_attention=return_attention,
        )

        if return_attention:
            attention_output, attention_weights = attention_result
        else:
            attention_output = attention_result

        tokens = tokens + attention_output
        tokens = tokens + self.mlp(self.mlp_norm(tokens))
        if return_attention:
            return tokens, attention_weights
        return tokens

# Application 1 · An autoregressive language model (4–25 min)

## 2 · The training question is “what comes next?”

Our transparent teaching tokenizer maps **one character to one integer ID**.
Production language models commonly use subword tokens; do not transfer our character counts
to those models.

The local corpus contains short, generated descriptions of colored objects, movements and results.
It lets a small model learn visible patterns. It does not establish broad language understanding.

In [ ]:
from __future__ import annotations
# TinyLM and the character tokenizer: standalone copies of the reusable course components.
"""Small character-level language-model components for Day 7B."""

from collections.abc import Sequence

import torch
from torch import nn



class CharacterTokenizer:
    """A transparent one-character-to-one-token tokenizer."""

    def __init__(self, characters: Sequence[str]) -> None:
        self.characters = tuple(characters)
        if not self.characters:
            raise ValueError("characters must not be empty.")
        if any(len(character) != 1 for character in self.characters):
            raise ValueError("every vocabulary item must be one character.")
        if len(set(self.characters)) != len(self.characters):
            raise ValueError("characters must be unique.")

        self.character_to_id = {
            character: token_id
            for token_id, character in enumerate(self.characters)
        }

    @classmethod
    def from_text(cls, text: str) -> "CharacterTokenizer":
        """Build a deterministic vocabulary from all characters in text."""

        if not text:
            raise ValueError("text must not be empty.")
        return cls(sorted(set(text)))

    @property
    def vocabulary_size(self) -> int:
        return len(self.characters)

    def encode(self, text: str) -> list[int]:
        """Convert text to integer token IDs."""

        unknown = sorted(set(text).difference(self.character_to_id))
        if unknown:
            visible = ", ".join(repr(character) for character in unknown)
            raise ValueError(f"text contains characters outside the vocabulary: {visible}")
        return [self.character_to_id[character] for character in text]

    def decode(self, token_ids: Sequence[int] | torch.Tensor) -> str:
        """Convert integer token IDs back to text."""

        if isinstance(token_ids, torch.Tensor):
            token_ids = token_ids.detach().cpu().reshape(-1).tolist()

        decoded = []
        for token_id in token_ids:
            index = int(token_id)
            if not 0 <= index < self.vocabulary_size:
                raise ValueError(f"token ID {index} is outside the vocabulary.")
            decoded.append(self.characters[index])
        return "".join(decoded)


def sample_next_token_batch(
    token_ids: torch.Tensor,
    *,
    batch_size: int,
    context_length: int,
    generator: torch.Generator | None = None,
) -> tuple[torch.Tensor, torch.Tensor]:
    """Sample input windows and their one-position-shifted targets."""

    if token_ids.ndim != 1 or token_ids.dtype != torch.long:
        raise ValueError("token_ids must be a one-dimensional torch.long Tensor.")
    if batch_size <= 0 or context_length <= 0:
        raise ValueError("batch_size and context_length must be positive.")
    if token_ids.numel() <= context_length:
        raise ValueError("token_ids must be longer than context_length.")

    maximum_start = token_ids.numel() - context_length - 1
    starts = torch.randint(
        maximum_start + 1,
        (batch_size,),
        generator=generator,
        device=token_ids.device,
    )
    offsets = torch.arange(context_length, device=token_ids.device)
    inputs = token_ids[starts[:, None] + offsets[None, :]]
    targets = token_ids[starts[:, None] + offsets[None, :] + 1]
    return inputs, targets


class TinyCausalLanguageModel(nn.Module):
    """A compact decoder-only Transformer that predicts the next token."""

    def __init__(
        self,
        *,
        vocabulary_size: int,
        max_sequence_length: int = 96,
        hidden_dim: int = 96,
        num_heads: int = 4,
        num_layers: int = 3,
        mlp_ratio: int = 4,
    ) -> None:
        super().__init__()
        if vocabulary_size <= 0 or max_sequence_length <= 0:
            raise ValueError(
                "vocabulary_size and max_sequence_length must be positive."
            )
        if num_layers <= 0:
            raise ValueError("num_layers must be positive.")

        self.vocabulary_size = vocabulary_size
        self.max_sequence_length = max_sequence_length
        self.hidden_dim = hidden_dim

        self.token_embedding = nn.Embedding(vocabulary_size, hidden_dim)
        self.position_embedding = nn.Parameter(
            torch.empty(1, max_sequence_length, hidden_dim)
        )
        nn.init.normal_(self.position_embedding, std=0.02)

        self.blocks = nn.ModuleList(
            [
                TransformerBlock(
                    hidden_dim,
                    num_heads,
                    mlp_ratio=mlp_ratio,
                )
                for _ in range(num_layers)
            ]
        )
        self.output_norm = nn.LayerNorm(hidden_dim)
        self.vocabulary_projection = nn.Linear(hidden_dim, vocabulary_size)

    def prepare_tokens(self, token_ids: torch.Tensor) -> torch.Tensor:
        """Create token representations with learned position information."""

        self._validate_token_ids(token_ids)
        sequence_length = token_ids.shape[1]
        return (
            self.token_embedding(token_ids)
            + self.position_embedding[:, :sequence_length]
        )

    def forward(
        self,
        token_ids: torch.Tensor,
        *,
        return_attention: bool = False,
    ) -> torch.Tensor | tuple[torch.Tensor, torch.Tensor]:
        tokens = self.prepare_tokens(token_ids)
        attention_maps = []

        for block in self.blocks:
            if return_attention:
                tokens, attention = block(
                    tokens,
                    causal=True,
                    return_attention=True,
                )
                attention_maps.append(attention)
            else:
                tokens = block(tokens, causal=True)

        logits = self.vocabulary_projection(self.output_norm(tokens))
        if return_attention:
            return logits, torch.stack(attention_maps, dim=1)
        return logits

    def _validate_token_ids(self, token_ids: torch.Tensor) -> None:
        if token_ids.ndim != 2:
            raise ValueError("token_ids must have shape [B, T].")
        if token_ids.dtype != torch.long:
            raise ValueError("token_ids must use torch.long dtype.")
        if token_ids.shape[1] == 0:
            raise ValueError("token sequence must not be empty.")
        if token_ids.shape[1] > self.max_sequence_length:
            raise ValueError(
                f"sequence length must not exceed {self.max_sequence_length}."
            )
        if torch.any(token_ids < 0) or torch.any(
            token_ids >= self.vocabulary_size
        ):
            raise ValueError("token_ids contain an ID outside the vocabulary.")

In [ ]:
def build_local_corpus():
    # Unique episodes, shuffled once and split by whole episode before token windows are sampled.
    rng = random.Random(7)
    names = ['ada', 'ben', 'cora', 'drew', 'emma', 'finn']
    colors = ['red', 'blue', 'green', 'white', 'yellow', 'purple']
    objects = ['cube', 'ball', 'cup', 'book', 'block', 'bowl']
    places = ['left tray', 'right tray', 'blue box', 'green box', 'top shelf', 'low table']
    keys = [(n, c, o, p) for n in names for c in colors for o in objects for p in places]
    rng.shuffle(keys)
    episodes = [
        f'task: move the {c} {o} to the {p}.\n'
        f'action: {n} carries the {c} {o} to the {p}.\n'
        f'result: the {c} {o} is on the {p}.\n\n'
        for n, c, o, p in keys
    ]
    split = int(.9 * len(episodes))
    return ''.join(episodes[:split]), ''.join(episodes[split:])


train_text, validation_text = build_local_corpus()
tokenizer = CharacterTokenizer.from_text(train_text)
train_ids = torch.tensor(tokenizer.encode(train_text), dtype=torch.long)
validation_ids = torch.tensor(tokenizer.encode(validation_text), dtype=torch.long)
CORPUS_HASH = hashlib.sha256((train_text + '\0' + validation_text).encode()).hexdigest()
print('One training episode:\n' + train_text.split('\n\n')[0])
print('\nVocabulary size:', tokenizer.vocabulary_size)
print('Training / validation characters:', len(train_ids), '/', len(validation_ids))
print('The validation episodes differ, but use the SAME sentence templates.')

**Predict before running:** for `move`, should the training pairs be `m→m`, `o→o`, `v→v`,
or should each input position predict the following character?

In [ ]:
SHIFT_TEXT = 'move the red'  # TRY another short phrase using the corpus vocabulary.
shift_ids = torch.tensor(tokenizer.encode(SHIFT_TEXT))
inputs, targets = shift_ids[:-1], shift_ids[1:]
if len(inputs) < 1:
    raise ValueError('Use at least two characters.')
fig, ax = plt.subplots(figsize=(12, 3.2))
ax.set_xlim(-1.4, len(inputs)-.4)
ax.set_ylim(-.65, 2.1)
ax.axis('off')
for i, (input_id, target_id) in enumerate(zip(inputs, targets)):
    for y, token_id, color in [(1.3, input_id, BLUE), (0, target_id, GREEN)]:
        ax.add_patch(Rectangle((i-.36, y-.28), .72, .56, fc=color, alpha=.15))
        char = visible_character(tokenizer.decode([token_id]))
        ax.text(i, y, char, ha='center', va='center', fontsize=10)
        ax.text(i, y-.43, f'ID {int(token_id)}', ha='center', fontsize=8)
    ax.annotate('', (i, .38), (i, .94), arrowprops={'arrowstyle': '->', 'color': PURPLE})
ax.text(-1.3, 1.3, 'Input', va='center')
ax.text(-1.3, 0, 'Target', va='center')
ax.set_title('Every position predicts the NEXT token; the target row is shifted by one')
fig.tight_layout()
plt.show()
print('Input IDs:', inputs.tolist())
print('Target IDs:', targets.tolist())

### New concept · Why must the model be prevented from seeing the answer? (4 min)

Consider the training text **`move`**:

| Input position | Input character | Target: next character | Context available when generating that target |
|---|---|---|---|
| 0 | `m` | `o` | `m` |
| 1 | `o` | `v` | `mo` |
| 2 | `v` | `e` | `mov` |

During training, the input tensor contains **`mov` all at once**. With unrestricted attention,
position 0 could read position 1 and copy `o`, its answer. But while generating from `m`,
that future `o` does not exist yet. This would teach the model to use information unavailable at generation time.

**Ask before running:** when position 1 (`o`) predicts `v`, which input characters should it be
allowed to read? Should it be allowed to read itself?

We impose a **causal mask**: position $i$ can read positions $0,\ldots,i$ only.
“Causal” here means later input positions cannot influence earlier predictions.
The animation contrasts unrestricted access with the new rule. Its matrix shows
**allowed connections (0 or 1), not learned attention weights**.

In [ ]:
def make_causal_mask_demo():
    from matplotlib.colors import ListedColormap
    inputs,targets=list('mov'),list('ove')
    fig,axes=plt.subplots(1,2,figsize=(11,4.8),gridspec_kw={'width_ratios':[1.6,1]})
    fig.subplots_adjust(top=.80,bottom=.22,wspace=.4)
    footer=fig.text(.5,.05,'',ha='center',fontsize=11)
    def draw(frame):
        query,masked=divmod(frame,2)
        allowed=np.tril(np.ones((3,3),dtype=int)) if masked else np.ones((3,3),dtype=int)
        for ax in axes:ax.clear()
        left,right=axes
        mode='Causal mask: future inputs blocked' if masked else 'Unrestricted attention: can read future inputs'
        fig.suptitle(mode,fontsize=15)
        left.set(xlim=(-.8,3),ylim=(-1.1,2));left.axis('off')
        left.text(-.6,1.6,'Training inputs (all already in the tensor)',fontsize=12)
        for j,char in enumerate(inputs):
            permitted=bool(allowed[query,j])
            color=GREEN if permitted else '#bcbcbc'
            left.add_patch(Rectangle((j-.3,.35),.6,.7,fc=color,alpha=.25,
                                     ec=ORANGE if j==query else color,lw=3 if j==query else 1))
            left.text(j,.72,char,ha='center',va='center',fontsize=22,color='black' if permitted else '#999999')
            left.text(j,.12,f'position {j}',ha='center',fontsize=10)
        visible=''.join(inputs[j] for j in range(3) if allowed[query,j])
        left.text(-.6,-.35,f'Position {query} reads: {visible!r}',fontsize=13)
        left.text(-.6,-.80,f'Target to predict: {targets[query]!r}',fontsize=13,color=PURPLE)
        right.imshow(allowed,cmap=ListedColormap(['#eeeeee','#b8ddcf']),vmin=0,vmax=1)
        for i in range(3):
            for j in range(3):right.text(j,i,str(allowed[i,j]),ha='center',va='center',fontsize=16)
        right.add_patch(Rectangle((-.5,query-.5),3,1,fill=False,ec=ORANGE,lw=3))
        labels=['0: m','1: o','2: v']
        right.set(xticks=range(3),xticklabels=labels,yticks=range(3),yticklabels=labels,
                  xlabel='Position being read (key)',ylabel='Position making prediction (query)',title='1 = allowed; 0 = blocked')
        if masked:message='The current character is allowed: its target is the NEXT character.'
        elif query<2:message=f"Leak: the answer {targets[query]!r} is already visible at future input position {query+1}."
        else:message="The final target 'e' is outside this input window; there is no later input to copy here."
        footer.set_text(message)
    return animation_html(fig,draw,frames=6,interval=1600)

display(make_causal_mask_demo())

### How the rule is implemented

Set future **scores** to $-\infty$ **before softmax**. Their exponentials become zero, so they
receive zero attention weight. Softmax then renormalizes over the permitted positions.
Multiplying by zero after softmax alone would not perform this renormalization.

The diagonal remains allowed: when predicting what follows `o`, `o` is known input, not the target `v`.

In [ ]:
# One illustrative score row for position 1: it can read m and o, but not future v.
scores = torch.tensor([.2, 1.3, .8])
query_position = 1
future_positions = torch.arange(len(scores)) > query_position
masked_scores = scores.masked_fill(future_positions, -torch.inf)
print('Original scores:       ', scores.tolist())
print('Scores before softmax: ', masked_scores.tolist())
print('Weights without mask:  ', scores.softmax(-1).numpy().round(3))
print('Weights with mask:     ', masked_scores.softmax(-1).numpy().round(3))
assert masked_scores.softmax(-1)[2] == 0

### Parallel training does not require access to future tokens

All input positions are already present in a training example, so all prediction rows can
be computed **in parallel**. Each row reads only its own allowed prefix.
During generation, future input tokens do not exist yet: we must select and append a token
before predicting the one after it.

```text
input IDs [B, T]
  → embeddings + positions [B, T, D]
  → causal Transformer blocks [B, T, D]
  → vocabulary projection [B, T, V]
```

There are **B × T next-token classification targets**, not just one per sequence.
Cross entropy compares each position's vocabulary scores with its actual next token.

In [ ]:
LM_CONFIG = dict(vocabulary_size=tokenizer.vocabulary_size, max_sequence_length=96,
                 hidden_dim=96, num_heads=4, num_layers=3, mlp_ratio=4)


def load_lm_checkpoint(path=LM_PATH):
    bundle = torch.load(path, map_location='cpu', weights_only=True)
    if bundle.get('schema') != SCHEMA or bundle.get('kind') != 'tiny_lm':
        raise ValueError('This is not a Tutorial 7 TinyLM checkpoint.')
    if bundle['config'] != LM_CONFIG or bundle['characters'] != list(tokenizer.characters):
        raise ValueError('Model configuration or tokenizer does not match this notebook.')
    if bundle['corpus_hash'] != CORPUS_HASH:
        raise ValueError('Checkpoint corpus differs from the notebook corpus.')
    model = TinyCausalLanguageModel(**bundle['config'])
    model.load_state_dict(bundle['state_dict'], strict=True)
    return model.eval(), bundle


torch.manual_seed(42)
lm = TinyCausalLanguageModel(**LM_CONFIG).eval()
lm_bundle = None
LM_STATUS = 'UNTRAINED — computation demo only'
if LM_PATH.exists():
    lm, lm_bundle = load_lm_checkpoint()
    LM_STATUS = 'PRETRAINED TinyLM'
print(LM_STATUS, '| parameters:', f'{count_parameters(lm):,}')

batch_x, batch_y = sample_next_token_batch(train_ids, batch_size=2, context_length=12,
                                           generator=torch.Generator().manual_seed(10))
with torch.inference_mode():
    batch_logits = lm(batch_x)
    batch_loss = F.cross_entropy(batch_logits.reshape(-1, tokenizer.vocabulary_size), batch_y.reshape(-1))
print('Inputs [B,T]:', tuple(batch_x.shape), '| Targets [B,T]:', tuple(batch_y.shape))
print('Logits [B,T,V]:', tuple(batch_logits.shape))
print(f'{batch_y.numel()} supervised positions; cross entropy = {float(batch_loss):.3f}')

## 3 · Watch text generation, one token at a time

### Two-minute website activity

Open **[Transformer Explainer](https://poloclub.github.io/transformer-explainer/)**.
Enter a short prompt such as `The cat sat on the`, inspect its token boundaries and the
next-token candidates, then generate a continuation. Change temperature and observe the distribution.
Do not spend class time tracing every internal matrix.

**Checkpoint question:** does one forward pass create the entire continuation?

If the website is slow or unavailable, continue with the local animation below.
For a deeper, optional view of the model's internal computation, explore
[LLM Visualization](https://bbycroft.net/llm) after class.

### The local generation loop

1. Encode the current text.
2. Run the model and take the vocabulary scores at the **last input position**.
3. Choose one next token and append it.
4. Repeat using the updated text.

The code below performs actual model calls. The animation replays the recorded probabilities
and decisions; animation playback does not rerun the model.

In [ ]:
def decoding_probabilities(logits, temperature=1.0, top_k=None):
    if temperature <= 0:
        raise ValueError('Temperature must be > 0; use greedy=True for argmax decoding.')
    scores = logits.detach().float().cpu() / temperature
    if top_k is not None:
        if not isinstance(top_k, int) or not 1 <= top_k <= scores.numel():
            raise ValueError('top_k must be an integer between 1 and vocabulary size.')
        _, indices = scores.topk(top_k)
        filtered = torch.full_like(scores, -torch.inf)
        filtered[indices] = scores[indices]
        scores = filtered
    return scores.softmax(-1)


@torch.inference_mode()
def generate_text(model, prompt, *, max_new_tokens=60, greedy=False,
                  temperature=1.0, top_k=None, seed=0, trace_steps=12):
    if not prompt:
        raise ValueError('Please provide a nonempty prompt.')
    if max_new_tokens < 0:
        raise ValueError('max_new_tokens must be nonnegative.')
    model.eval()
    device = next(model.parameters()).device
    ids = torch.tensor([tokenizer.encode(prompt)], dtype=torch.long, device=device)
    rng = torch.Generator().manual_seed(seed)  # Sampling on CPU for a reproducible local experiment.
    trace = []
    for step in range(max_new_tokens):
        context = ids[:, -model.max_sequence_length:]
        next_logits = model(context)[0, -1].float().cpu()
        # In greedy mode the chart shows the unmodified model probabilities.
        probabilities = decoding_probabilities(next_logits, 1.0 if greedy else temperature,
                                                None if greedy else top_k)
        next_id = int(next_logits.argmax()) if greedy else int(torch.multinomial(probabilities, 1, generator=rng))
        if step < trace_steps:
            trace.append({'prefix': tokenizer.decode(ids[0]), 'context': tokenizer.decode(context[0]),
                          'probabilities': probabilities.clone(), 'next_id': next_id})
        ids = torch.cat([ids, torch.tensor([[next_id]], device=device)], dim=1)
    return tokenizer.decode(ids[0]), trace

In [ ]:
def make_generation_demo(trace, status):
    if not trace:
        raise ValueError('Record at least one generation step.')
    fig = plt.figure(figsize=(12, 5.5))
    ax_text = fig.add_axes([.05, .17, .47, .65])
    ax_probs = fig.add_axes([.62, .2, .34, .57])
    fig.text(.5, .045, 'Predict → choose → append → run the model again. Each frame is one recorded model call.', ha='center', fontsize=10)

    def draw(i):
        ax_text.clear(); ax_probs.clear()
        item = trace[i]
        fig.suptitle(f'{status}\nGeneration step {i+1}', fontsize=14)
        ax_text.axis('off')
        prefix = item['prefix']
        shown = prefix[-140:].replace('\n', r'\n')
        ax_text.text(0, 1, 'Current text', va='top', weight='bold')
        ax_text.text(0, .86, textwrap.fill(shown, 45), va='top', family='monospace', fontsize=11)
        next_char = tokenizer.decode([item['next_id']])
        ax_text.text(0, .40, f'Append: {visible_character(next_char)!r}', color=GREEN, fontsize=16)
        ax_text.text(0, .20, 'Next model input includes this new character.', fontsize=10)
        if len(prefix) > len(item['context']):
            ax_text.text(0, .04, f'Only the last {len(item["context"])} characters fit in the context window.', fontsize=9)
        p = item['probabilities']
        candidate_ids = p.topk(min(6, len(p))).indices.tolist()
        if item['next_id'] not in candidate_ids:
            candidate_ids[-1] = item['next_id']
        labels = [visible_character(tokenizer.decode([j])) for j in candidate_ids]
        ax_probs.barh(range(len(candidate_ids)), p[candidate_ids].numpy(),
                      color=[GREEN if j == item['next_id'] else BLUE for j in candidate_ids])
        ax_probs.set(yticks=range(len(candidate_ids)), yticklabels=labels, xlim=(0, 1.05),
                     xlabel='Probability used for this decision', title='Candidates (selected token in green)')
        ax_probs.invert_yaxis()
        for row, j in enumerate(candidate_ids):
            ax_probs.text(float(p[j])+.015, row, f'{p[j]:.2f}', va='center', fontsize=9)
    return animation_html(fig, draw, len(trace), interval=900)

In [ ]:
# TRY: keep the prompt fixed first. Use only characters in the local vocabulary.
PROMPT = 'task: move the red cube'
generated_text, generation_trace = generate_text(lm, PROMPT, max_new_tokens=48,
                                                 greedy=True, trace_steps=12)
print(LM_STATUS)
print(repr(generated_text))
display(make_generation_demo(generation_trace, LM_STATUS))

## 4 · Same model, different decoding choices

Temperature rescales the same logits before softmax:
$$p_i(T)=\frac{\exp(z_i/T)}{\sum_j\exp(z_j/T)}.$$

- **Lower positive temperature:** a more concentrated distribution.
- **Higher temperature:** a flatter distribution.
- **Greedy:** always choose the largest logit; no sampling randomness.
- **Optional top-k:** retain only the k highest-scoring candidates before sampling.

These settings do **not** update any model weights. Higher temperature does not add knowledge
or guarantee better output. Greedy is not guaranteed to produce the best entire sequence.

**Predict first:** if temperature changes, does the highest-scoring token change?
What happens to the probability assigned to lower-scoring tokens?

In [ ]:
# First isolate the effect using FIVE hand-chosen logits (not predictions from a trained model).
example_words = ['mat', 'sofa', 'floor', 'roof', 'moon']
example_logits = torch.tensor([3., 2., 1., 0., -1.])
TEMPERATURES = [0.5, 1.0, 1.8]  # TRY changing one value; all must be positive.
fig, axes = plt.subplots(1, len(TEMPERATURES), figsize=(12, 3.5), sharey=True, squeeze=False)
for ax, temp in zip(axes.flat, TEMPERATURES):
    p = decoding_probabilities(example_logits, temperature=temp)
    ax.bar(example_words, p.numpy(), color=BLUE)
    ax.set(title=f'T = {temp:g}', ylim=(0, 1), xlabel='Candidate token')
    for j in range(len(p)):
        ax.text(j, float(p[j])+.025, f'{p[j]:.2f}', ha='center', fontsize=9)
axes[0, 0].set_ylabel('Probability')
fig.suptitle('Same illustrative logits; only temperature changes')
fig.tight_layout()
plt.show()

In [ ]:
# TRY: compare the SAME prompt and seed. Then repeat with a second seed.
EXPERIMENT_PROMPT = 'task: move the '
SAMPLING_SEED = 123
print(LM_STATUS)
for label, options in [
    ('greedy', dict(greedy=True)),
    ('temperature 0.6', dict(temperature=.6)),
    ('temperature 1.0', dict(temperature=1.0)),
    ('temperature 1.6', dict(temperature=1.6)),
]:
    continuation, _ = generate_text(lm, EXPERIMENT_PROMPT, max_new_tokens=80,
                                    seed=SAMPLING_SEED, trace_steps=0, **options)
    print('\n' + label + ':\n' + continuation)

**Discuss with a neighbor (one minute):**
1. Which outputs resemble the training templates? Which break words or repeat themselves?
2. Once the first sampled token differs, are later predictions still conditioned on the same text?
3. Did changing temperature train a new model?

If you see **UNTRAINED**, discuss only the mechanics of sampling. Use the TA's checkpoint
to assess learned text patterns. One prompt and one seed do not establish a universal “best” temperature.

**Transition:** now change tasks. We will first understand diffusion, then identify the job
assigned to the Transformer inside that model.

# Application 2 · A Transformer inside image diffusion (25–42 min)

## 5 · From random noise to a new image

### First, what are we trying to achieve? (about 3 min for the introduction)

We want to **generate a new handwritten-digit image**, having learned from many real examples.
At generation time we supply random noise: no original digit image and no requested digit label.

The three pictures below are states from one saved generation run. A trained model repeatedly
helps update the image until a digit-like shape emerges. Some samples are imperfect.

**Look first:** what changes between the first and last picture? Where could the knowledge
of digit shapes come from if the starting image is just noise?

In [ ]:
# Preview saved generation states; no model loading, training or live sampling.
def make_diffusion_intro_preview(sample_index=0):
    if not DEMO_PATH.exists():
        print('The generation preview needs tutorial7_dit_demo.npz. Continue with the noising animation below.')
        return None
    with np.load(DEMO_PATH, allow_pickle=False) as saved:
        if saved['schema'].item() != SCHEMA or saved['model_kind'].item() != 'tiny_dit_epsilon_noskip_v1':
            raise ValueError('Use the matching Tutorial 7 noise-prediction demo.')
        if DIT_PATH.exists() and file_sha256(DIT_PATH) != saved['checkpoint_sha256'].item():
            raise ValueError('The preview cache and TinyDiT checkpoint differ.')
        seeds = saved['seeds']
        if not 0 <= sample_index < len(seeds):
            raise ValueError('Choose one of the saved initial-noise seeds.')
        states = saved['trajectory'][:, sample_index, 0].copy()
        seed = int(seeds[sample_index])
        times = saved['trajectory_times'].copy()
    if states.ndim != 3 or states.shape[1:] != (28, 28) or len(states) < 3 or not np.isfinite(states).all():
        raise ValueError('Invalid saved generation states.')
    if len(times) != len(states) or times[-1] != -1:
        raise ValueError('The preview must show a complete sampling run.')
    last = len(states)-1
    indices = [0, last//2, last]
    fig, axes = plt.subplots(1, 3, figsize=(10, 4))
    fig.subplots_adjust(top=.80, bottom=.18, wspace=.35)
    titles = ['Start: random noise', f'After {indices[1]} updates', f'After {last} updates: generated image']
    for ax, index, title in zip(axes, indices, titles):
        ax.imshow(states[index], cmap='gray', vmin=-1, vmax=1)
        ax.set_title(title, fontsize=11)
        ax.axis('off')
    fig.suptitle('The goal: generate a new image from noise', fontsize=15)
    fig.text(.37, .49, '→', ha='center', fontsize=24, color=BLUE)
    fig.text(.65, .49, '→', ha='center', fontsize=24, color=BLUE)
    fig.text(.5, .055, f'Saved outputs of the trained model · initial-noise seed {seed}\n'
             'No original digit image or digit label is supplied during generation.', ha='center', fontsize=10)
    return fig


diffusion_intro_figure = make_diffusion_intro_preview(sample_index=0)
if diffusion_intro_figure is not None:
    plt.show()

### Learn from a task we can construct ourselves

Start with real digit images and deliberately corrupt them with noise. Now the model can
practice **denoising**: using what it has learned about digits to estimate image structure
from noisy input. We train at many noise levels, from almost clean to almost pure noise.

At high noise, the original image cannot be recovered with certainty. The model has to rely
on patterns learned across many training examples.

The animation below shows how we create training inputs. We choose the clean image,
noise and noise level ourselves, so we can construct training examples automatically.
Watch the digit become harder to recognize as the noise level increases.

In [ ]:
NUM_DIFFUSION_STEPS = 100


def cosine_alpha_bar(num_steps=NUM_DIFFUSION_STEPS):
    # Discrete cosine schedule with beta clipping; index t runs from 0 to num_steps-1.
    grid = torch.linspace(0, num_steps, num_steps+1, dtype=torch.float64)
    curve = torch.cos(((grid / num_steps + .008) / 1.008) * math.pi / 2).square()
    curve = curve / curve[0]
    beta = (1 - curve[1:] / curve[:-1]).clamp(0, .999)
    return torch.cumprod(1-beta, dim=0).float()


ALPHA_BAR = cosine_alpha_bar()


def add_noise(clean, noise, timesteps, alpha_bar=ALPHA_BAR):
    a = alpha_bar.to(clean.device)[timesteps].view(-1, 1, 1, 1)
    return a.sqrt() * clean + (1-a).sqrt() * noise


def make_noising_demo(digit_index=3):
    x0 = clean_images[digit_index:digit_index+1]
    epsilon = torch.randn(x0.shape, generator=torch.Generator().manual_seed(18))
    levels = [-1, 0, 9, 24, 49, 74, 89, 99]  # -1 is a display-only clean endpoint.
    fig, axes = plt.subplots(1, 3, figsize=(10, 4.3))
    fig.subplots_adjust(bottom=.22, top=.80)
    footer = fig.text(.5, .055, '', ha='center', fontsize=10)

    def draw(frame):
        t = levels[frame]
        a = torch.tensor(1.) if t == -1 else ALPHA_BAR[t]
        xt = x0 if t == -1 else add_noise(x0, epsilon, torch.tensor([t]))
        for ax, values, title in zip(axes, [x0, epsilon, xt],
                                      ['Clean image x₀', 'Sampled noise ε', 'Constructed input xₜ']):
            ax.clear()
            ax.imshow(values[0, 0], cmap='gray', vmin=-1, vmax=1)
            ax.set_title(title); ax.axis('off')
        fig.suptitle('Training-input construction — not learned image generation', fontsize=14)
        footer.set_text(f'{"Clean endpoint" if t == -1 else f"Noise level t = {t}"}   |   '
                        f'signal coefficient = {a.sqrt():.3f}   noise coefficient = {(1-a).sqrt():.3f}\n'
                        'The SAME ε is held fixed to isolate noise level; each frame uses the marginal formula.\n'
                        'Display clips values outside [-1, 1]; the actual noisy tensor is not clipped.')
    return animation_html(fig, draw, len(levels), interval=950)

display(make_noising_demo())

### How do we train the denoiser? Predict the noise we added

Why predict noise when our goal is an image? **An estimate of the added noise lets us
estimate the clean image.** Noise prediction gives the network a concrete training task,
with a target that we already know because we sampled it ourselves.

For a clean training image $x_0$, sample Gaussian noise $\epsilon$ and a noise level $t$:
$$x_t = a_t x_0 + b_t\epsilon,\qquad
a_t=\sqrt{\bar\alpha_t},\quad b_t=\sqrt{1-\bar\alpha_t}.$$
Here $a_t$ and $b_t$ control how much image and noise are mixed together.

The Transformer receives $x_t$ and $t$, and predicts $\hat\epsilon=\epsilon_\theta(x_t,t)$:
$$\mathcal L=\operatorname{MSE}(\hat\epsilon,\epsilon).$$

```text
clean image + sampled noise + noise level → noisy training input
noisy image + noise level → Transformer → predicted noise
loss = ordinary MSE(predicted noise, the noise we actually added)
```

All pixels have equal weight in the loss. The network sees **only the noisy image and
noise level**; the clean image constructs the training input, and the sampled noise supplies
the training target. Neither the clean image nor the target noise is supplied to the network.

**Pause:** do we need a human to label this training target? Why does the network also
need to know the noise level?

### Connect training to generation

| During training | During generation |
|---|---|
| Start with a real image and deliberately add noise | Start with random noise; no original image is given |
| Know the sampled noise and use it to compute the loss | Use the model's noise estimate; no target answer is available |
| Update model parameters using many examples and noise levels | Keep parameters fixed and repeatedly update the current image |

For generation, the same trained model is called repeatedly:

1. Predict noise $\hat\epsilon$ from the current image and noise level.
2. Estimate the clean image: $\hat x_0=(x_t-b_t\hat\epsilon)/a_t$.
3. Let the sampler move to a lower noise level, then repeat with the updated image.

A prediction is imperfect, especially at high noise. The sampler makes a sequence of
updates; it does not simply subtract the entire predicted noise once and stop.

**This is not a replay of one training image's corruption in reverse.** During generation
there is no original image to recover. The model uses the patterns it learned to help form
a new sample. We will return to the full generation animation after seeing how images
become Transformer tokens.

**Diffusion does not specify one network architecture.** In this notebook, the learned
noise predictor is a Transformer.

## 6 · How can a Transformer accept an image?

Split a `28 × 28` image into `2 × 2` patches, in row-major order:

```text
[B, 1, 28, 28] → [B, 196, 4] → linear embedding → [B, 196, D]
```

There are `14 × 14 = 196` image tokens; each patch initially contains `4` pixel values.
Patchify only rearranges pixels. Embedding changes their representation; position information
records where each patch belongs. The animation below shows the rearrangement **before embedding**.

**Predict first:** after patch 13 (the end of the first row), where does patch 14 come from?

In [ ]:
def make_patch_demo(digit_index=3, patch_size=2):
    image = clean_images[digit_index:digit_index+1]
    patches = patchify(image, patch_size)
    restored = unpatchify(patches, image_size=28, patch_size=patch_size)
    assert torch.equal(restored, image)
    side = 28 // patch_size
    # Selected positions keep the animation short while showing the row boundary.
    selected = [0, 1, 2, side-1, side, side+1, patches.shape[1]//2, patches.shape[1]-1]
    fig, axes = plt.subplots(1, 3, figsize=(12, 4.6), gridspec_kw={'width_ratios': [1, 1.8, 1]})
    fig.subplots_adjust(top=.80, bottom=.28, wspace=.4)
    fig.suptitle('Image → ordered patches → reconstructed image', fontsize=15)
    fig.text(.5, .045, 'Middle column: one token. Middle row: one pixel component within that patch.\n'
             'This is pixel rearrangement, not compression or a learned embedding.', ha='center', fontsize=10)

    def draw(frame):
        index = selected[frame]
        row, col = divmod(index, side)
        for ax in axes:
            ax.clear()
        for ax, pixels, title in [(axes[0], image[0, 0], 'Original image'),
                                  (axes[2], restored[0, 0], 'Unpatchify: exact reconstruction')]:
            ax.imshow(pixels, cmap='gray', vmin=-1, vmax=1)
            for boundary in np.arange(-.5, 28, patch_size):
                ax.axhline(boundary, color=BLUE, lw=.4)
                ax.axvline(boundary, color=BLUE, lw=.4)
            ax.add_patch(Rectangle((col*patch_size-.5, row*patch_size-.5), patch_size, patch_size,
                                   ec=ORANGE, fill=False, lw=3))
            ax.set_title(title, fontsize=11); ax.axis('off')
        axes[1].imshow(patches[0].T, cmap='gray', vmin=-1, vmax=1, aspect='auto')
        axes[1].add_patch(Rectangle((index-.5, -.5), 1, patch_size**2, fill=False, ec=ORANGE, lw=2))
        axes[1].set(title=f'Token {index}: image row {row}, column {col}',
                    xlabel=f'Token index (0–{patches.shape[1]-1})',
                    ylabel=f'Pixel component (0–{patch_size**2-1})',
                    xticks=sorted(set([0, side, patches.shape[1]//2, patches.shape[1]-1])))
    return animation_html(fig, draw, len(selected), interval=850)

display(make_patch_demo())

### Give the Transformer a specific prediction task

Our image Transformer uses **196 image tokens plus one timestep token**. It adds
position information and supplies timestep context to the patch embeddings and output head.

```text
noisy image → patch embeddings ─┐
timestep → timestep token ──────┴→ 197 tokens → full-attention Transformer
                                               ↓ keep 196 image-token outputs
                                     predict 4 noise values per patch
                                               ↓ unpatchify
                                     predicted noise [B, 1, 28, 28]
```

The output head receives **Transformer features and timestep context only**.
It has no direct shortcut from the original noisy pixel values. The noisy image still
enters normally through the patch embeddings.

All noisy patches exist at this step, so full attention is appropriate. Raster order
does not require the patches to be generated from left to right.

In [ ]:
from __future__ import annotations
# TinyDiT: standalone adaptation of the course repository model.
"""A compact patch-token diffusion Transformer for Day 7."""

import torch
from torch import nn



class TinyDiffusionTransformer(nn.Module):
    """Predict diffusion noise from noisy patches and a timestep token; no raw-pixel shortcut at the head."""

    def __init__(
        self,
        *,
        image_size: int = 28,
        patch_size: int = 2,
        channels: int = 1,
        hidden_dim: int = 96,
        num_heads: int = 4,
        num_layers: int = 4,
        num_timesteps: int = 100,
        mlp_ratio: int = 4,
    ) -> None:
        super().__init__()
        if image_size % patch_size:
            raise ValueError("image_size must be divisible by patch_size.")
        if num_layers <= 0 or num_timesteps <= 0:
            raise ValueError("num_layers and num_timesteps must be positive.")

        self.image_size = image_size
        self.patch_size = patch_size
        self.channels = channels
        self.hidden_dim = hidden_dim
        self.num_timesteps = num_timesteps
        self.patch_count = (image_size // patch_size) ** 2
        self.patch_values = channels * patch_size * patch_size
        self.condition_token_count = 1
        self.sequence_length = self.condition_token_count + self.patch_count

        self.patch_projection = nn.Linear(self.patch_values, hidden_dim)
        self.timestep_embedding = nn.Embedding(num_timesteps, hidden_dim)
        self.timestep_projection = nn.Sequential(
            nn.Linear(hidden_dim, hidden_dim),
            nn.SiLU(),
            nn.Linear(hidden_dim, hidden_dim),
        )
        self.position_embedding = nn.Parameter(
            torch.empty(1, self.sequence_length, hidden_dim)
        )
        nn.init.normal_(self.position_embedding, std=0.02)

        self.blocks = nn.ModuleList(
            [
                TransformerBlock(
                    hidden_dim,
                    num_heads,
                    mlp_ratio=mlp_ratio,
                )
                for _ in range(num_layers)
            ]
        )
        self.output_norm = nn.LayerNorm(hidden_dim)
        self.noise_projection = nn.Sequential(
            nn.Linear(2 * hidden_dim, hidden_dim),
            nn.GELU(),
            nn.Linear(hidden_dim, self.patch_values),
        )

    @property
    def is_conditional(self) -> bool:
        return False

    def prepare_tokens(
        self,
        noisy_images: torch.Tensor,
        timesteps: torch.Tensor,
    ) -> torch.Tensor:
        """Create `[time, image patches]` tokens with position information."""

        self._validate_inputs(noisy_images, timesteps)
        noisy_patches = patchify(noisy_images, self.patch_size)
        timestep_embedding = self.timestep_embedding(timesteps)
        timestep_context = self.timestep_projection(timestep_embedding)
        image_tokens = self.patch_projection(noisy_patches)
        image_tokens = image_tokens + timestep_context.unsqueeze(1)
        tokens = torch.cat([timestep_embedding.unsqueeze(1), image_tokens], dim=1)
        return tokens + self.position_embedding

    def forward(
        self,
        noisy_images: torch.Tensor,
        timesteps: torch.Tensor,
    ) -> torch.Tensor:
        self._validate_inputs(noisy_images, timesteps)
        timestep_context = self.timestep_projection(
            self.timestep_embedding(timesteps)
        ).unsqueeze(1)
        tokens = self.prepare_tokens(noisy_images, timesteps)
        for block in self.blocks:
            tokens = block(tokens)

        image_tokens = self.output_norm(tokens[:, self.condition_token_count :])
        decoder_tokens = torch.cat(
            [
                image_tokens,
                timestep_context.expand(-1, self.patch_count, -1),
            ],
            dim=-1,
        )
        predicted_noise_patches = self.noise_projection(decoder_tokens)
        return unpatchify(
            predicted_noise_patches,
            image_size=self.image_size,
            patch_size=self.patch_size,
            channels=self.channels,
        )

    def _validate_inputs(
        self,
        noisy_images: torch.Tensor,
        timesteps: torch.Tensor,
    ) -> None:
        if noisy_images.ndim != 4:
            raise ValueError("noisy_images must have shape [B, C, H, W].")
        expected_image_shape = (
            noisy_images.shape[0],
            self.channels,
            self.image_size,
            self.image_size,
        )
        if tuple(noisy_images.shape) != expected_image_shape:
            raise ValueError(
                "noisy_images must have shape "
                f"[B, {self.channels}, {self.image_size}, {self.image_size}]."
            )
        batch_size = noisy_images.shape[0]
        if timesteps.shape != (batch_size,):
            raise ValueError("timesteps must contain one index per image.")
        if timesteps.dtype != torch.long:
            raise ValueError("timesteps must use torch.long dtype.")

In [ ]:
DIT_CONFIG = dict(image_size=28, patch_size=2, channels=1, hidden_dim=96,
                  num_heads=4, num_layers=4, num_timesteps=NUM_DIFFUSION_STEPS, mlp_ratio=4)


def load_dit_checkpoint(path=DIT_PATH):
    bundle = torch.load(path, map_location='cpu', weights_only=True)
    if bundle.get('schema') != SCHEMA or bundle.get('kind') != 'tiny_dit_epsilon_noskip_v1' or bundle.get('prediction_type') != 'epsilon':
        raise ValueError('Use the updated noise-prediction Tutorial 7 checkpoint with this notebook.')
    if bundle['config'] != DIT_CONFIG or bundle['preprocessing'] != 'uint8_div127.5_minus1':
        raise ValueError('TinyDiT configuration or preprocessing mismatch.')
    if not torch.allclose(bundle['alpha_bar'], ALPHA_BAR, rtol=1e-6, atol=1e-8):
        raise ValueError('Diffusion schedule mismatch.')
    model = TinyDiffusionTransformer(**bundle['config'])
    model.load_state_dict(bundle['state_dict'], strict=True)
    return model.eval(), bundle


torch.manual_seed(43)
dit = TinyDiffusionTransformer(**DIT_CONFIG).eval()
dit_bundle = None
DIT_STATUS = 'UNTRAINED — shape demonstration only'
if DIT_PATH.exists():
    dit, dit_bundle = load_dit_checkpoint()
    DIT_STATUS = 'PRETRAINED TinyDiT'
example_t = torch.tensor([50])
example_noise = torch.randn((1, 1, 28, 28), generator=torch.Generator().manual_seed(12))
example_noisy = add_noise(clean_images[3:4], example_noise, example_t)
with torch.inference_mode():
    patch_values = patchify(example_noisy, dit.patch_size)
    prepared_tokens = dit.prepare_tokens(example_noisy, example_t)
    predicted_noise = dit(example_noisy, example_t)
    noise_mse = F.mse_loss(predicted_noise, example_noise)
assert predicted_noise.shape == example_noisy.shape
print(DIT_STATUS, '| parameters:', f'{count_parameters(dit):,}')
print('Noisy image [B,C,H,W]:', tuple(example_noisy.shape))
print('Raw patches [B,N,P²]: ', tuple(patch_values.shape))
print('With time token [B,N+1,D]:', tuple(prepared_tokens.shape))
print('Predicted noise [B,C,H,W]:', tuple(predicted_noise.shape))
print(f'Unweighted noise MSE for this one example: {float(noise_mse):.4f}')

## 7 · Watch a model actually generate images

The classroom default is **playback of saved sampling results**. Every saved state was
produced by the trained TinyDiT and the sampler below. Playback is immediate; it does not
train a model or perform 100 new network calls.

The file records the checkpoint hash, initial seeds, schedule, sampler settings and sample states.
If the file is absent, ask the TA for the prepared demo file or use the preparation appendix.
The add-noise animation above remains available, but is not a substitute for generated samples.

**Observe first:** when do strokes become visible? What changes in the later steps?
All image patches are updated on every sampling step.

In [ ]:
# Deterministic DDIM-style sampling from noise predictions, with clipped x0.
# Skipped timesteps are handled by the update formula; this is not a truncated 100-step loop.
@torch.inference_mode()
def sample_dit(model, initial_noise, *, calls=100, alpha_bar=ALPHA_BAR):
    if not isinstance(calls, int) or not 1 <= calls <= len(alpha_bar):
        raise ValueError('calls must be an integer from 1 to the schedule length.')
    model.eval()
    device = next(model.parameters()).device
    x = initial_noise.detach().clone().to(device)
    schedule = alpha_bar.to(device)
    times = torch.linspace(len(schedule)-1, 0, calls).round().long().tolist()
    states, state_times = [x.cpu().clone()], [times[0]]
    for position, t in enumerate(times):
        previous_t = times[position+1] if position+1 < len(times) else -1
        a = schedule[t]
        previous_a = schedule[previous_t] if previous_t >= 0 else torch.tensor(1., device=device)
        predicted_noise = model(x, torch.full((len(x),), t, dtype=torch.long, device=device))
        x0_estimate = ((x - (1-a).sqrt()*predicted_noise) / a.sqrt()).clamp(-1, 1)
        # Recompute the noise direction after clipping x0 to keep the pair consistent with x_t.
        direction = (x - a.sqrt() * x0_estimate) / (1-a).sqrt()
        x = previous_a.sqrt() * x0_estimate + (1-previous_a).sqrt() * direction
        if not torch.isfinite(x).all():
            raise RuntimeError(f'Nonfinite sample at timestep {t}.')
        states.append(x.cpu().clone())
        state_times.append(previous_t)
    return x.cpu(), torch.stack(states), np.asarray(state_times, dtype=np.int64)


def load_demo(path=DEMO_PATH):
    with np.load(path, allow_pickle=False) as source:
        data = {key: source[key].copy() for key in source.files}
    if data['schema'].item() != SCHEMA or data['sampler'].item() != 'ddim_epsilon_eta0_clip_x0_consistent_eps':
        raise ValueError('Unrecognized demo format or sampler.')
    if data['model_kind'].item() != 'tiny_dit_epsilon_noskip_v1' or data['model_config'].item() != str(sorted(DIT_CONFIG.items())):
        raise ValueError('Demo model configuration does not match this notebook.')
    if not np.allclose(data['alpha_bar'], ALPHA_BAR.numpy()):
        raise ValueError('Demo schedule does not match this notebook.')
    frames = data['trajectory']
    expected_shape = (len(data['trajectory_times']), len(data['seeds']), 1, 28, 28)
    if frames.shape != expected_shape or not np.isfinite(frames).all():
        raise ValueError('Invalid trajectory dimensions or values.')
    if data['finals'].shape != (len(data['calls']), len(data['seeds']), 1, 28, 28):
        raise ValueError('Invalid comparison image dimensions.')
    if not np.isfinite(data['finals']).all():
        raise ValueError('Nonfinite comparison images.')
    if DIT_PATH.exists() and file_sha256(DIT_PATH) != data['checkpoint_sha256'].item():
        raise ValueError('The demo and local TinyDiT checkpoint differ. Re-export the demo.')
    return data


def make_sampling_demo(data, sample_index=0):
    if not 0 <= sample_index < len(data['seeds']):
        raise ValueError('Choose a sample index listed in the demo file.')
    states = data['trajectory'][:, sample_index, 0]
    times = data['trajectory_times']
    frame_indices = np.unique(np.linspace(0, len(states)-1, 21).round().astype(int))
    final = states[-1]
    fig, axes = plt.subplots(1, 3, figsize=(10, 4.4))
    fig.subplots_adjust(top=.78, bottom=.2)
    footer = fig.text(.5, .055, '', ha='center', fontsize=10)

    def draw(frame):
        index = frame_indices[frame]
        for ax, pixels, title in zip(axes, [states[0], states[index], final],
                                     ['Initial random noise', 'Current sample state', 'Final generated sample']):
            ax.clear(); ax.imshow(pixels, cmap='gray', vmin=-1, vmax=1)
            ax.set_title(title, fontsize=11); ax.axis('off')
        fig.suptitle(f'PRETRAINED TinyDiT · saved real sampling trajectory · seed {data["seeds"][sample_index]}', fontsize=13)
        level = 'clean endpoint' if times[index] == -1 else f'noise level t={times[index]}'
        footer.set_text(f'{index} / {len(states)-1} network calls completed; {level}\n'
                        'Each update changes the WHOLE image. Display clips values outside [-1, 1].')
    return animation_html(fig, draw, len(frame_indices), interval=450)

In [ ]:
diffusion_demo = load_demo() if DEMO_PATH.exists() else None
SAMPLE_INDEX = 0  # TRY 0, 1, 2 or 3 after the TA exports the four-seed demo.
if diffusion_demo is not None:
    print('Checkpoint SHA-256:', diffusion_demo['checkpoint_sha256'].item())
    print('Available initial-noise seeds:', diffusion_demo['seeds'].tolist())
    display(make_sampling_demo(diffusion_demo, SAMPLE_INDEX))
else:
    print('No pretrained sampling demo found:', DEMO_PATH)
    print('TA: train/load TinyDiT and export the demo in the appendix, then rerun this cell.')
    print('No generated image is being claimed or displayed here.')

### Controlled experiment: 3, 5 or 10 network calls

**Before revealing the grid:** what do you expect to change when the same model gets
more denoising updates, starting from exactly the same initial noise?

Each row completes a sampling run with **3, 5 or 10 calls**. All runs reach the clean endpoint;
these are not the first few frames of the 100-call animation above.

The sampler visits these noise levels before its final update to the clean endpoint:

- **3 calls:** `[99, 50, 0]`
- **5 calls:** `[99, 74, 50, 25, 0]`
- **10 calls:** `[99, 88, 77, 66, 55, 44, 33, 22, 11, 0]`

More calls visit more intermediate noise levels, with smaller gaps between them.
Inspect the actual images: which call count gives recognizable strokes? How much changes
from 3 to 5 calls, and from 5 to 10? Judge the observed results for this model and sampler;
more calls do not guarantee a better image for every seed.

The animation and broader gallery still use 100 calls. This model is unconditional:
a random seed controls the initial noise; it is not a requested digit label.

In [ ]:
if diffusion_demo is not None:
    comparison = torch.from_numpy(diffusion_demo['finals'])
    titles = [f'{calls} {"call" if calls == 1 else "calls"} · seed {seed}' for calls in diffusion_demo['calls']
              for seed in diffusion_demo['seeds']]
    image_grid(comparison.flatten(0, 1), titles, columns=len(diffusion_demo['seeds']),
               heading='Same pretrained model; same initial noise within each column')
else:
    print('Comparison grid needs tutorial7_dit_demo.npz; no training or sampling starts automatically.')

### Broader check: all fixed seeds 100–115

These 16 seeds were chosen as a consecutive range before viewing their samples.
No images in this grid are omitted or replaced. Identify recognizable digits **and** ambiguous
examples. This is visual inspection, not a measurement of digit-classification accuracy.

In [ ]:
if diffusion_demo is not None:
    image_grid(diffusion_demo['gallery_samples'],
               [f"seed {seed}" for seed in diffusion_demo['gallery_seeds']], columns=8,
               heading='Noise-prediction Transformer · fixed seeds 100–115 · 100 calls')
else:
    print('The updated tutorial7_dit_demo.npz contains this gallery.')

**Discuss:** compare across columns (different initial noise) and down rows (different call counts).
Why might jumping directly to the clean endpoint leave a poor estimate? Are all generated shapes recognizable?
Decreasing a training loss alone is not enough to establish image quality.

### Optional live sample · skip if class is running behind

Only enable this after loading the pretrained TinyDiT weights. It generates one new sample
using 20 calls. CPU speed varies; the saved experiment above is the classroom default.

In [ ]:
RUN_LIVE_DIFFUSION = True
LIVE_SEED = 42
if RUN_LIVE_DIFFUSION:
    if dit_bundle is None:
        raise RuntimeError('Load tutorial7_dit.pt first; live generation requires pretrained weights.')
    live_noise = torch.randn((1, 1, 28, 28), generator=torch.Generator().manual_seed(LIVE_SEED))
    start = time.perf_counter()
    live_final, _, _ = sample_dit(dit, live_noise, calls=20)
    print(f'20 network calls took {time.perf_counter()-start:.2f} seconds on CPU.')
    image_grid(live_final, [f'PRETRAINED · seed {LIVE_SEED}'], columns=1)
else:
    print('Live diffusion sampling disabled; use the saved trajectories above.')

## 8 · Check understanding (42–45 min)

Fill this in before opening the answer. These are the **two particular models in this notebook**,
not universal rules for every language or image model.

| Question | Our autoregressive LM | Our diffusion Transformer |
|---|---|---|
| What is one input token? | ? | ? |
| Which positions can attention read? | ? | ? |
| What does the network predict? | ? | ? |
| What is the training target / loss? | ? | ? |
| What changes after one inference iteration? | ? | ? |

<details>
<summary>Reveal the comparison</summary>

| Question | Our autoregressive LM | Our diffusion Transformer |
|---|---|---|
| Input token | Character | Noisy image patch, plus a timestep token |
| Allowed attention | Current and earlier positions | All noisy-image patches and timestep token |
| Prediction | Vocabulary logits at each position | Noise values for every image patch |
| Target / loss | Next character / cross entropy | Sampled Gaussian noise / ordinary MSE |
| Inference iteration | Append one selected token | Update the whole image toward a lower noise level |

</details>

**Exit questions:**
1. Why can our LM train on many positions in parallel but generate new tokens sequentially?
2. Does a higher temperature change the model weights?
3. Why does our TinyDiT not need a causal mask? Is diffusion itself a Transformer architecture?

<details>
<summary>Brief answer checks</summary>

1. Training knows every target and uses a causal mask for every row. At generation time,
   the next input prefix depends on the token just selected.
2. No. It changes how logits become sampling probabilities.
3. Every current noisy patch is already available. Diffusion specifies a training/generation
   framework; the prediction network can use different architectures.

</details>

## Optional extension · How many image tokens? (3–5 min)

For a square image with side length $H$ and patch size $P$:
$$N_{\rm image}=(H/P)^2.$$

Our TinyDiT adds one timestep token. One attention head has an
$(N_{\rm image}+1)\times(N_{\rm image}+1)$ weight matrix.

**Predict first:** if we halve the patch width, do we double or quadruple the number of
image tokens? What happens to the number of pairwise attention weights?

This counts attention-matrix entries, **not total model runtime**. Changing patch size also
changes patch projection/output dimensions and requires a compatible model and checkpoint.

In [ ]:
print(f'{"Patch":<10}{"Image tokens":>15}{"Total tokens":>15}{"Weights/head":>17}')
for patch_size in [7, 4, 2, 1]:
    image_tokens = (28 // patch_size) ** 2
    total_tokens = image_tokens + 1
    print(f'{patch_size} × {patch_size:<6}{image_tokens:>15,}{total_tokens:>15,}{total_tokens**2:>17,}')
print('\nFrom 4×4 to 2×2: 4× as many image tokens; approximately 16× the attention entries.')
print('The time token makes the total-token ratio slightly different.')

# Preparation appendix · TA before class / students after class

**Outside the 45-minute lesson. All switches default to False.**

1. Run the notebook definitions. On this Mac, keep the default **CPU** training device below.
   CUDA is selected automatically when available. This base environment (PyTorch 2.0.1)
   crashed inside the MPS backend during a short TinyLM test; CPU completed the same test.
   After a kernel crash, restart the kernel and rerun the definitions before training.
   Device changes are read when training/export is called, or pass `device="cpu"` explicitly.
2. Enable **only** `TRAIN_LM` to train/save TinyLM, then turn it back off.
3. Enable **only** `TRAIN_DIT` to train/save TinyDiT, then turn it back off.
4. Enable `EXPORT_DIFFUSION_DEMO` to generate the real 3/5/10-call comparison and separate 100-call trajectory cache.
5. Reload the classroom checkpoint cells and inspect text and images. If samples are poor,
   improve training before class; do not present denoising-target loss as proof of image quality.
6. Reset all switches to False and save the notebook with its useful outputs.

Training does not overwrite existing output files. To train another version, use a new path.
The two checkpoints also save configuration, training history and preprocessing/tokenizer metadata.
The demo is tied to the TinyDiT file by SHA-256.

The supplied model is trained **from random initialization**: 2×2 patches, 4 Transformer layers,
hidden dimension 96, batch size 128, and 12,000 updates at constant learning rate `1e-3`.
It predicts noise with ordinary, equally weighted pixel MSE. The output head receives
Transformer features and timestep context; it has no raw noisy-patch shortcut.
EMA weights (decay 0.995) are used for sampling.
TinyLM validation uses new episodes from the same templates, not a general language benchmark.
TinyDiT uses 59,000 MNIST training images and a disjoint 1,000-image validation subset;
the embedded test images are never used by the optimizer.

In [ ]:
# CPU is the stable default on this Mac (base: PyTorch 2.0.1).
# MPS reproduced a native process crash during TinyLM training on this environment.
# CUDA is still selected automatically on supported machines.
TRAIN_DEVICE = 'cuda' if torch.cuda.is_available() else 'cpu'
print('Optional training device:', TRAIN_DEVICE)


def save_new_checkpoint(bundle, destination):
    # Exclusive creation prevents accidental overwrite of a trained model.
    path = Path(destination)
    with path.open('xb') as stream:
        torch.save(bundle, stream)


def cpu_state(model):
    return {name: value.detach().cpu().clone() for name, value in model.state_dict().items()}


@torch.inference_mode()
def estimate_lm_loss(model, token_ids, *, seed=900, batches=8, batch_size=32):
    model.eval()
    device = next(model.parameters()).device
    rng = torch.Generator().manual_seed(seed)
    losses = []
    for _ in range(batches):
        x, y = sample_next_token_batch(token_ids, batch_size=batch_size,
                                       context_length=model.max_sequence_length, generator=rng)
        logits = model(x.to(device))
        loss = F.cross_entropy(logits.reshape(-1, tokenizer.vocabulary_size), y.to(device).reshape(-1))
        losses.append(float(loss))
    return float(np.mean(losses))


def train_lm_checkpoint(destination=LM_PATH, *, steps=1500, batch_size=32, learning_rate=3e-4,
                        device=None, evaluation_interval=100, seed=42):
    device = TRAIN_DEVICE if device is None else device
    if Path(destination).exists():
        raise FileExistsError(f'Refusing to overwrite {destination}. Choose a new path.')
    if steps < 1 or batch_size < 1 or evaluation_interval < 1:
        raise ValueError('steps, batch_size and evaluation_interval must be positive.')
    torch.manual_seed(seed)
    model = TinyCausalLanguageModel(**LM_CONFIG).to(device)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate)
    rng = torch.Generator().manual_seed(seed+1)
    history = [{'step': 0, 'train_loss': estimate_lm_loss(model, train_ids),
                'validation_loss': estimate_lm_loss(model, validation_ids)}]
    best_loss, best_state, best_step = float('inf'), None, 0
    started = time.perf_counter()
    for step in range(1, steps+1):
        model.train()
        x, y = sample_next_token_batch(train_ids, batch_size=batch_size,
                                       context_length=model.max_sequence_length, generator=rng)
        logits = model(x.to(device))
        loss = F.cross_entropy(logits.reshape(-1, tokenizer.vocabulary_size), y.to(device).reshape(-1))
        if not torch.isfinite(loss):
            raise RuntimeError(f'Nonfinite language loss at step {step}.')
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.0)
        optimizer.step()
        if step == 1 or step % evaluation_interval == 0 or step == steps:
            train_loss = estimate_lm_loss(model, train_ids)
            val_loss = estimate_lm_loss(model, validation_ids)
            history.append({'step': step, 'train_loss': train_loss, 'validation_loss': val_loss})
            print(f'step {step:4d}/{steps} | train {train_loss:.3f} | validation {val_loss:.3f}', flush=True)
            if val_loss < best_loss:
                best_loss, best_state, best_step = val_loss, cpu_state(model), step
    bundle = dict(schema=SCHEMA, kind='tiny_lm', config=LM_CONFIG, characters=list(tokenizer.characters),
                  corpus_hash=CORPUS_HASH, state_dict=best_state, history=history,
                  selected_step=best_step, training=dict(steps=steps, batch_size=batch_size,
                  learning_rate=learning_rate, seed=seed, device=str(device)),
                  elapsed_seconds=time.perf_counter()-started)
    save_new_checkpoint(bundle, destination)
    print('Saved:', destination, '| selected validation checkpoint at step', best_step)
    return bundle

In [ ]:
TRAIN_LM = False
if TRAIN_LM:
    trained_lm_bundle = train_lm_checkpoint(steps=1500)
    print('Next: set TRAIN_LM=False, rerun the TinyLM loading cell and the generation experiments.')
else:
    print('TinyLM training disabled.')

In [ ]:
@torch.inference_mode()
def estimate_dit_loss(model, validation_images, *, seed=901, batch_size=64):
    """Ordinary noise MSE on a fixed validation noise/timestep draw."""
    model.eval()
    device = next(model.parameters()).device
    rng = torch.Generator().manual_seed(seed)
    total, denominator = 0., 0.
    for start in range(0, len(validation_images), batch_size):
        x0 = validation_images[start:start+batch_size]
        noise = torch.randn(x0.shape, generator=rng)
        t = torch.randint(NUM_DIFFUSION_STEPS, (len(x0),), generator=rng)
        prediction = model(add_noise(x0, noise, t).to(device), t.to(device)).cpu()
        total += float((prediction-noise).square().sum())
        denominator += noise.numel()
    return total / denominator


def train_dit_checkpoint(destination=DIT_PATH, *, steps=12000, batch_size=128,
                         learning_rate=1e-3, ema_decay=.995,
                         device=None, seed=42, data_directory='data',
                         evaluation_interval=2000):
    """Train a fresh, unconditional epsilon predictor with ordinary noise MSE."""
    device = TRAIN_DEVICE if device is None else device
    if Path(destination).exists():
        raise FileExistsError(f'Refusing to overwrite {destination}. Choose a new path.')
    if min(steps, batch_size, evaluation_interval) < 1 or not 0 <= ema_decay < 1:
        raise ValueError('Positive training sizes and 0 <= ema_decay < 1 are required.')
    from torchvision.datasets import MNIST
    dataset = MNIST(data_directory, train=True, download=True)
    permutation = torch.randperm(len(dataset), generator=torch.Generator().manual_seed(43))
    validation_indices = permutation[20000:21000]
    eligible = torch.ones(len(dataset), dtype=torch.bool)
    eligible[validation_indices] = False
    train_indices = torch.where(eligible)[0]
    pixels = (dataset.data[train_indices, None].float()/127.5 - 1).to(device)
    validation_pixels = dataset.data[validation_indices, None].float()/127.5 - 1
    torch.manual_seed(seed)
    model = TinyDiffusionTransformer(**DIT_CONFIG)
    model = model.to(device)
    ema = copy.deepcopy(model).eval().requires_grad_(False)
    optimizer = torch.optim.AdamW(model.parameters(), lr=learning_rate, weight_decay=.01)
    torch.manual_seed(seed+1)
    rng = torch.Generator().manual_seed(seed+1)
    history = []
    started = time.perf_counter()
    for step in range(1, steps+1):
        model.train()
        indices = torch.randint(len(pixels), (batch_size,), generator=rng).to(device)
        x0 = pixels[indices]
        noise = torch.randn_like(x0)
        t = torch.randint(NUM_DIFFUSION_STEPS, (len(x0),), device=device)
        a = ALPHA_BAR.to(device)[t].view(-1, 1, 1, 1)
        noisy = a.sqrt()*x0 + (1-a).sqrt()*noise
        prediction = model(noisy, t)
        loss = F.mse_loss(prediction, noise)
        if not torch.isfinite(loss):
            raise RuntimeError(f'Nonfinite noise loss at step {step}.')
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1.)
        optimizer.step()
        with torch.no_grad():
            for average, current in zip(ema.parameters(), model.parameters()):
                average.lerp_(current, 1-ema_decay)
        if step % 500 == 0:
            print(f'step {step}/{steps} | noise MSE {float(loss.detach()):.4f}', flush=True)
        if step % evaluation_interval == 0 or step == steps:
            metric = estimate_dit_loss(ema, validation_pixels)
            history.append({'step': step, 'validation_loss': metric})
            print(f'EMA validation noise MSE: {metric:.4f}', flush=True)
    bundle = dict(schema=SCHEMA, kind='tiny_dit_epsilon_noskip_v1', config=DIT_CONFIG,
                  prediction_type='epsilon', preprocessing='uint8_div127.5_minus1',
                  alpha_bar=ALPHA_BAR.clone(), state_dict=cpu_state(ema), history=history,
                  selected_step=steps, train_indices=train_indices, validation_indices=validation_indices,
                  training=dict(steps=steps, batch_size=batch_size, learning_rate=learning_rate,
                                final_learning_rate=learning_rate, ema_decay=ema_decay,
                                foreground_extra_weight=0., seed=seed, sampling_seed=seed+1, device=str(device),
                                initialization='random; no pretrained weights',
                                output_raw_patch_shortcut=False, loss='unweighted noise MSE',
                                learning_rate_schedule='constant', weight_decay=.01, gradient_clip_norm=1.),
                  elapsed_seconds=time.perf_counter()-started)
    save_new_checkpoint(bundle, destination)
    print('Saved EMA noise-prediction weights:', destination)
    return bundle

In [ ]:
TRAIN_DIT = False
if TRAIN_DIT:
    trained_dit_bundle = train_dit_checkpoint(steps=12000)  # Fresh initialization; constant learning rate 1e-3.
    print('Next: set TRAIN_DIT=False, then export the classroom demo below.')
else:
    print('TinyDiT training disabled; no dataset download.')

In [ ]:
def export_diffusion_demo(checkpoint_path=DIT_PATH, destination=DEMO_PATH, *, device=None):
    device = TRAIN_DEVICE if device is None else device
    if Path(destination).exists():
        raise FileExistsError(f'Refusing to overwrite {destination}. Choose a new path.')
    model, bundle = load_dit_checkpoint(checkpoint_path)
    model = model.to(device)
    seeds = np.asarray([11, 22, 33, 44], dtype=np.int64)
    initial = torch.stack([torch.randn((1, 28, 28), generator=torch.Generator().manual_seed(int(seed)))
                           for seed in seeds])
    call_counts = [3, 5, 10]
    finals, timings = [], []
    for calls in call_counts:
        started = time.perf_counter()
        final, trajectory, times = sample_dit(model, initial, calls=calls)
        timings.append(time.perf_counter()-started)
        finals.append(final.numpy())
        print(f'Exported {calls:3d} calls × {len(seeds)} samples in {timings[-1]:.2f}s', flush=True)
    # The comparison has fewer calls; separately retain the full animation trajectory.
    _, trajectory, times = sample_dit(model, initial, calls=100)
    # Keep the original four classroom seeds; add an uncurated 16-seed gallery.
    gallery_seeds = np.arange(100, 116, dtype=np.int64)
    gallery_noise = torch.stack([torch.randn((1, 28, 28), generator=torch.Generator().manual_seed(int(seed)))
                                 for seed in gallery_seeds])
    gallery_samples, _, _ = sample_dit(model, gallery_noise, calls=100)
    # trajectory/times still refer to the original four seeds at 100 calls.
    with Path(destination).open('xb') as stream:
        np.savez_compressed(stream, schema=np.asarray(SCHEMA), model_kind=np.asarray('tiny_dit_epsilon_noskip_v1'),
            model_config=np.asarray(str(sorted(DIT_CONFIG.items()))),
            sampler=np.asarray('ddim_epsilon_eta0_clip_x0_consistent_eps'),
            checkpoint_sha256=np.asarray(file_sha256(checkpoint_path)),
            alpha_bar=ALPHA_BAR.numpy(), seeds=seeds, calls=np.asarray(call_counts),
            initial_noise=initial.numpy(), finals=np.stack(finals),
            gallery_seeds=gallery_seeds, gallery_samples=gallery_samples.numpy(),
            trajectory=trajectory.numpy(), trajectory_times=times, trajectory_calls=np.asarray(100),
            elapsed_seconds=np.asarray(timings), export_device=np.asarray(str(device)))
    print('Saved:', destination)
    return Path(destination)

In [ ]:
EXPORT_DIFFUSION_DEMO = False
if EXPORT_DIFFUSION_DEMO:
    export_diffusion_demo()
    print('Next: set EXPORT_DIFFUSION_DEMO=False and rerun the sampling playback/comparison cells.')
else:
    print('Demo export disabled; no background diffusion sampling.')

### Inspect training histories after loading the checkpoints

These plots show optimization and held-out prediction loss. Also inspect generated text and
images in the classroom sections. The checkpoint selected by validation loss need not have
the best subjective generations.

In [ ]:
available = [(name, bundle, x_key) for name, bundle, x_key in
             [('TinyLM', lm_bundle, 'step'), ('TinyDiT', dit_bundle, 'step')] if bundle is not None]
if not available:
    print('No trained histories loaded. Train separately, then rerun the classroom loading cells.')
else:
    fig, axes = plt.subplots(1, len(available), figsize=(6*len(available), 3.5), squeeze=False)
    for ax, (name, bundle, x_key) in zip(axes.flat, available):
        history = bundle['history']
        for key, label in [('train_loss', 'Training'), ('validation_loss', 'Validation')]:
            records = [r for r in history if key in r]
            ax.plot([r[x_key] for r in records], [r[key] for r in records], marker='.', label=label)
        ax.set(title=name, xlabel=x_key, ylabel='Cross entropy' if name == 'TinyLM' else 'Noise MSE')
        ax.legend()
    fig.tight_layout()
    plt.show()

### Files to distribute after preparation

For the main classroom activities, provide this notebook, **`tutorial7_lm.pt`** and
**`tutorial7_dit_demo.npz`**. Include **`tutorial7_dit.pt`** if students should generate
new images locally. The notebook already contains the model definitions and example pixels.
No standalone HTML, repository clone or helper `.py` file is required.

Before uploading, ensure the four switches `TRAIN_LM`, `TRAIN_DIT`,
`EXPORT_DIFFUSION_DEMO` and `RUN_LIVE_DIFFUSION` are **False**. Turn `UPLOAD_FILES` off too.
Save useful classroom outputs after checking that they are labelled with the correct model status.